# 01 · Frame: Fairfare (Task A, The Fare)
**Owner: P1** · reviewed by P2, P3 · Status: `[x]`

Purpose: the question, the moment of prediction, the target's decisions, the baseline ladder. This notebook is the narrative version of `PROBLEM.md`; keep the two consistent.

## 1. The question
> Given the **pickup time, pickup zone, drop-off zone and planned route length** known at booking, predict the meter's **`fare_amount`** for a **standard metered (Rate #01) yellow-taxi trip in New York City**, so that an app can show a quote *before* the trip starts that the final meter fare stays under on 90% of trips.

## 2. Moment of prediction: booking
Known: pickup time, pickup/drop-off zone, planned route length (`trip_distance`).
Not known: drop-off time, duration, tip, total, tolls and any fee settled at the end, `RatecodeID`, `payment_type`.

**Why `trip_distance` is an assumption.** In the data it is the odometer distance *measured during the trip*. A booking app only has a routed estimate, which is wrong whenever the driver detours or the route changes. We use the measured value as a stand-in for the planned one, so our errors are optimistic by the routing error. NB04 §2b measures how fast accuracy degrades when distance is noisy.

**Two layers of protection against leakage:**
1. `FORBIDDEN` (config) lists every post-trip column; `assert_no_leak` rejects any feature set that names one.
2. `BOOKING_INPUTS` lists the *only* raw columns the model pipeline is given. `make_pipeline` selects them before any feature is computed, so a feature cannot be derived from a post-trip column even by accident. A test (`tests/test_protocol.py`) scrambles every forbidden column and checks the predictions do not change.

In [1]:
import sys; sys.path.insert(0, "..")          # repo root, so `import src...` works
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.config import TARGET
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 140)

from src.config import FORBIDDEN, BOOKING_INPUTS
from src.features import FEATURE_SETS, assert_no_leak
for name, cols in FEATURE_SETS.items():
    assert_no_leak(cols)                      # raises if any feature is not knowable at booking
assert not set(BOOKING_INPUTS) & FORBIDDEN
print("All feature sets respect the booking-time whitelist:", list(FEATURE_SETS))
print("Pipeline reads only:", BOOKING_INPUTS)

All feature sets respect the booking-time whitelist: ['f0_distance', 'f1_time', 'f2_geo', 'f3_airport', 'f4_shape', 'f5_zones', 'f6_pair', 'f7_time_x']
Pipeline reads only: ('tpep_pickup_datetime', 'PULocationID', 'DOLocationID', 'trip_distance', 'pickup_borough', 'dropoff_borough')


## 3. First look at the raw data (all months, nothing dropped)

In [2]:
from src.data import load_raw
raw = load_raw()
print(raw.shape)
display(raw.dtypes.to_frame("dtype").T)
display(raw["payment_type"].value_counts(dropna=False).sort_index().to_frame("trips").assign(share=lambda d: (d.trips / d.trips.sum()).round(3)))
display(raw["RatecodeID"].value_counts(dropna=False).sort_index().to_frame("trips").assign(share=lambda d: (d.trips / d.trips.sum()).round(3)))

(600000, 25)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,request_source,pickup_borough,pickup_zone,dropoff_borough,dropoff_zone
dtype,int32,datetime64[us],datetime64[us],float64,float64,float64,str,int32,int32,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,str,str,str,str,str


,trips,share
payment_type,,
0,154475,0.257
1,383942,0.640
2,56414,0.094
3,1723,0.003
4,3446,0.006


,trips,share
RatecodeID,,
1.0,412451,0.687
2.0,14264,0.024
3.0,1844,0.003
4.0,1361,0.002
5.0,5646,0.009
99.0,9959,0.017
NaN,154475,0.257


**Read it.** 600 000 trips (≈200 000 per month). A quarter (25.7%) have `payment_type = 0` — TLC's **flex fare** code — and exactly those rows have no `RatecodeID` (the 154 475 `NaN`s). Of the metered rows, 68.7% are Rate #01 (standard); the rest are JFK flat (2), Newark (3), Nassau/Westchester (4), negotiated (5) and `99` (unknown). Only standard-rate, card/cash trips are the population we price; NB02 justifies each cut with evidence.

## 4. Target decisions
From the `clean()` log in NB02 (May). Removals are sequential: each row counts the trips removed *by that step*, after the steps above it.

| # | Decision | Rows removed | % of May input | Why |
|---|---|---|---|---|
| 1 | Unknown/missing zones (264, 265) | 1 271 | 0.64 | no usable location |
| 2 | Flex fare (payment_type 0) | 46 429 | 23.22 | app/upfront priced, not the meter's time+distance process |
| 3 | Payment not card/cash | 1 650 | 0.83 | no charge / dispute / unknown / voided are not real fares |
| 4 | Non-standard rate code | 13 493 | 6.75 | JFK flat, Newark, negotiated, unknown (99) follow other rules |
| 5 | `fare_amount <= 0` | 195 | 0.10 | not a fare |
| 6 | `trip_distance <= 0` | 725 | 0.36 | no route |
| 7 | Duration < 1 min or > 180 min | 2 729 | 1.37 | corrupt timestamps (validity check only) |
| 8 | Above May 99.9% caps (24.85 mi, $96.80) | 165 | 0.08 | caps fitted on May only |

133 342 of 199 999 May trips remain (66.7%). Two pickups dated April 2026 are dropped before the split.

## 5. Baseline ladder
> **Not pre-registered.** The template asked for expected values *before* running NB04, but the skeleton run on 2026-11-01 executed the ladder before anyone wrote expectations down. Writing "expectations" now would be post hoc, so we only record the reasoning and the actual June numbers (NB04 §1).

| Rung | Model | Reasoning | Actual June MAE |
|---|---|---|---|
| 1 | May median fare | ignores distance; error ≈ spread of fares | 7.90 |
| 2 | start + per-mile rule (fit on May) | distance explains R² = 0.88 of fare (NB02 A1) | 2.98 |
| 3 | depth-2 decision tree | 4 distance buckets; coarser than a line | 3.95 |